# MALTO — v7: ModernBERT-base Conservative (Anti-Overfitting Fix)

**Goal:** Fix the OOF→Public gap in solution.ipynb (was 0.9532 OOF → 0.92643 public).

**Root cause of gap:**
1. Temperature search [0.3, 3.0] → found T=0.30 (boundary = OOF overfit)
2. Threshold sweep [0.60, 1.50] passes=8 → DeepSeek=1.485, Gemini=1.454 (aggressive overfit)
3. Per-class Nelder-Mead blend → extra overfit layer

**Fixes applied:**
1. Temperature: search [0.5, 2.0] only
2. Threshold: conservative [0.85, 1.20] passes=5 (original winning range)
3. Global blend only (no per-class Nelder-Mead)
4. 3-fold instead of 5-fold (~90 min instead of ~150 min)

**Runtime:** ~90 min on T4×2


In [ ]:
import os, warnings, gc, time, re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler

from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack as sparse_hstack
from scipy.optimize import minimize
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

SEED = 42
NUM_LABELS = 6
LABEL_MAP  = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    torch.cuda.manual_seed_all(SEED)
    N_GPUS  = torch.cuda.device_count()
    USE_AMP = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32       = False
    print(f'GPU: {torch.cuda.get_device_name(0)} x {N_GPUS}')
else:
    DEVICE = torch.device('cpu'); N_GPUS = 1; USE_AMP = False

KAGGLE_PATHS = [
    '/kaggle/input/malto-recruitment-hackathon',
    '/kaggle/input/test-and-train',
    '/kaggle/input/datasets/aliivaezii/test-and-train',
    '/kaggle/input', '.', '..'
]
DATA_DIR = None
for p in KAGGLE_PATHS:
    if os.path.exists(os.path.join(p, 'train.csv')):
        DATA_DIR = p; break
if DATA_DIR is None and os.path.isdir('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.csv' in files:
            DATA_DIR = root; break
assert DATA_DIR is not None

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
train_df['TEXT'] = train_df['TEXT'].fillna('empty')
test_df['TEXT']  = test_df['TEXT'].fillna('empty')

y_all       = train_df['LABEL'].values
texts_train = train_df['TEXT'].values
texts_test  = test_df['TEXT'].values

print(f'Device: {DEVICE} | GPUs: {N_GPUS}')
print(f'Train: {len(train_df)}, Test: {len(test_df)}')
t_start = time.time()


In [ ]:
# ============================================================
# CONFIG
# ============================================================
MODEL_NAME  = 'answerdotai/ModernBERT-base'
MAX_LEN     = 512
BATCH_SIZE  = 16 * N_GPUS
EPOCHS      = 10
LR          = 3e-5
PATIENCE    = 2
N_FOLDS     = 3         # 3-fold for ~90 min runtime
NUM_WORKERS = 4
DRW_CAP     = 20        # 20x — from winning solution, needed for DeepSeek weight
TOP_K_CKPTS = 3
FULL_EPOCHS = 5         # full-data model epochs

print(f'Model:   {MODEL_NAME}')
print(f'Batch:   {BATCH_SIZE} ({BATCH_SIZE//N_GPUS}/GPU)  |  LR: {LR}  |  Folds: {N_FOLDS}')
print(f'DRW cap: {DRW_CAP}x  |  SWA top-{TOP_K_CKPTS}  |  Patience: {PATIENCE}')


In [ ]:
# ============================================================
# LOSS + DATASET + OPTIMIZER
# ============================================================
class LDAMLoss(nn.Module):
    def __init__(self, class_counts, max_margin=0.5,
                 drw_epoch=2, drw_ramp_epochs=3, label_smoothing=0.1):
        super().__init__()
        safe = np.maximum(class_counts, 1).astype(np.float64)
        margins = np.clip(max_margin / np.power(safe, 0.25), 0, 1.0)
        self.register_buffer('margins', torch.tensor(margins, dtype=torch.float32))
        self.drw_epoch = drw_epoch; self.drw_ramp_epochs = drw_ramp_epochs
        self.current_epoch = 0; self.label_smoothing = label_smoothing
        inv = 1.0 / safe; w = inv / inv.sum() * len(class_counts)
        w   = np.clip(w, w.min(), w.min() * DRW_CAP)
        w   = w / w.sum() * len(class_counts)
        w   = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
        self.register_buffer('drw_weights', torch.tensor(w, dtype=torch.float32))

    def set_epoch(self, e): self.current_epoch = e

    def _blended_weight(self, device):
        ep = self.current_epoch
        if ep < self.drw_epoch: return None
        t = min((ep - self.drw_epoch) / max(self.drw_ramp_epochs, 1), 1.0)
        ones = torch.ones_like(self.drw_weights)
        return (ones + t * (self.drw_weights - ones)).to(device)

    def forward(self, logits, targets):
        logits = logits.float()
        if torch.isnan(logits).any(): logits = torch.nan_to_num(logits, nan=0.0)
        margins  = self.margins.to(logits.device)[targets]
        adjusted = logits.clone()
        adjusted[torch.arange(len(targets), device=logits.device), targets] -= margins
        weight = self._blended_weight(logits.device)
        loss   = F.cross_entropy(adjusted, targets, weight=weight,
                                 label_smoothing=self.label_smoothing)
        if torch.isnan(loss): return F.cross_entropy(logits, targets)
        return loss


class TextDataset(Dataset):
    def __init__(self, enc, labels=None, indices=None):
        self.enc = enc; self.labels = labels; self.indices = indices
    def __len__(self):
        return len(self.indices) if self.indices is not None else len(self.enc['input_ids'])
    def __getitem__(self, idx):
        i = self.indices[idx] if self.indices is not None else idx
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item


def get_optimizer(model, lr, llrd=0.9):
    no_decay = ['bias', 'LayerNorm', 'layernorm', 'layer_norm']
    params   = []
    head = [(n, p) for n, p in model.named_parameters()
            if 'classifier' in n or 'head' in n]
    if head:
        params += [{'params': [p for n, p in head if not any(nd in n for nd in no_decay)],
                    'lr': lr, 'weight_decay': 0.01},
                   {'params': [p for n, p in head if     any(nd in n for nd in no_decay)],
                    'lr': lr, 'weight_decay': 0.0}]
    layer_nums = set()
    for n, _ in model.named_parameters():
        m = re.search(r'(?:encoder\.layer|layers)\.(\d+)\.', n)
        if m: layer_nums.add(int(m.group(1)))
    num_layers = max(layer_nums) + 1 if layer_nums else 22
    for i in range(num_layers - 1, -1, -1):
        layer_lr = lr * (llrd ** (num_layers - 1 - i))
        lp = [(n, p) for n, p in model.named_parameters()
              if any(f'encoder.layer.{i}.' in n or f'layers.{i}.' in n for _ in [None])]
        lp = [(n, p) for n, p in model.named_parameters()
              if f'encoder.layer.{i}.' in n or f'layers.{i}.' in n]
        if lp:
            params += [{'params': [p for n, p in lp if not any(nd in n for nd in no_decay)],
                        'lr': layer_lr, 'weight_decay': 0.01},
                       {'params': [p for n, p in lp if     any(nd in n for nd in no_decay)],
                        'lr': layer_lr, 'weight_decay': 0.0}]
    assigned = set(id(p) for g in params for p in g['params'])
    rem = [p for n, p in model.named_parameters() if id(p) not in assigned and p.requires_grad]
    if rem: params.append({'params': rem, 'lr': lr * 0.5, 'weight_decay': 0.01})
    return torch.optim.AdamW(params)

print('Loss / Dataset / Optimizer ready.')


In [ ]:
# ============================================================
# TOKENIZATION
# ============================================================
print(f'Tokenizing with {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_enc = tokenizer(list(texts_train), max_length=MAX_LEN, padding='max_length',
                      truncation=True, return_tensors='pt')
test_enc  = tokenizer(list(texts_test),  max_length=MAX_LEN, padding='max_length',
                      truncation=True, return_tensors='pt')
print(f'Train: {train_enc["input_ids"].shape}, Test: {test_enc["input_ids"].shape}')


In [ ]:
# ============================================================
# K-FOLD TRAINING  (LDAM + SWA top-K)
# ============================================================
def train_fold(fold_idx, train_idx, val_idx):
    gc.collect(); torch.cuda.empty_cache()
    print(f'\n{"="*50}')
    print(f'FOLD {fold_idx+1}/{N_FOLDS} | Train: {len(train_idx)}, Val: {len(val_idx)}')
    print(f'{"="*50}')

    train_loader = DataLoader(TextDataset(train_enc, y_all, train_idx),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=True, drop_last=True)
    val_loader  = DataLoader(TextDataset(train_enc, y_all, val_idx),
        batch_size=BATCH_SIZE*2, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(TextDataset(test_enc),
        batch_size=BATCH_SIZE*2, num_workers=NUM_WORKERS, pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
    # Gradient checkpointing works correctly for ModernBERT
    base = model.model if hasattr(model, 'model') else model
    base.gradient_checkpointing_enable()
    if N_GPUS > 1: model = nn.DataParallel(model)
    model.to(DEVICE)

    fold_counts = np.bincount(y_all[train_idx], minlength=NUM_LABELS).astype(float)
    criterion   = LDAMLoss(fold_counts)
    criterion.to(DEVICE)
    print(f'  DRW weights: {criterion.drw_weights.cpu().numpy().round(2)}')

    actual_model = model.module if hasattr(model, 'module') else model
    optimizer = get_optimizer(actual_model, lr=LR, llrd=0.9)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    scaler = GradScaler(enabled=USE_AMP)

    top_ckpts = []; best_f1 = 0; patience_ctr = 0

    for epoch in range(EPOCHS):
        t0 = time.time()
        criterion.set_epoch(epoch)
        drw_pct = '  0%' if epoch < criterion.drw_epoch else \
            f'{int(min((epoch-criterion.drw_epoch)/criterion.drw_ramp_epochs,1)*100):3d}%'

        model.train()
        total_loss, valid_steps = 0.0, 0
        for batch in tqdm(train_loader, desc=f'E{epoch+1}', leave=False):
            optimizer.zero_grad()
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')
            with autocast(enabled=USE_AMP):
                out  = model(**inputs)
                loss = criterion(out.logits, labels)
            if torch.isnan(loss) or torch.isinf(loss):
                scheduler.step(); continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            total_loss += loss.item(); valid_steps += 1
            scheduler.step()

        model.eval()
        preds, lbls = [], []
        with torch.no_grad():
            for batch in val_loader:
                inp = {k: v.to(DEVICE) for k, v in batch.items()}
                lbl = inp.pop('labels')
                with autocast(enabled=USE_AMP): out = model(**inp)
                preds.extend(out.logits.argmax(-1).cpu().numpy())
                lbls.extend(lbl.cpu().numpy())

        val_f1 = f1_score(lbls, preds, average='macro')
        cur_m  = model.module if hasattr(model, 'module') else model
        state  = {k: v.cpu().clone() for k, v in cur_m.state_dict().items()}
        if len(top_ckpts) < TOP_K_CKPTS or val_f1 > top_ckpts[-1][0]:
            if len(top_ckpts) >= TOP_K_CKPTS: top_ckpts.pop()
            top_ckpts.append((val_f1, state))
            top_ckpts.sort(key=lambda x: x[0], reverse=True)

        if val_f1 > best_f1:
            best_f1 = val_f1; patience_ctr = 0; marker = ' ** BEST **'
        else:
            patience_ctr += 1; marker = f' (p={patience_ctr}/{PATIENCE})'
        print(f'  E{epoch+1} | loss={total_loss/max(valid_steps,1):.4f} | '
              f'val_f1={val_f1:.4f} | DRW={drw_pct} | {time.time()-t0:.0f}s{marker}')
        if patience_ctr >= PATIENCE: print(f'  Early stop at E{epoch+1}'); break

    # SWA: average top-K checkpoints
    n_avg = len(top_ckpts)
    print(f'  SWA top-{n_avg}: {[f"{f:.4f}" for f,_ in top_ckpts]}')
    avg_state = {k: (sum(c[k].float() for _,c in top_ckpts)/n_avg
                     ).to(top_ckpts[0][1][k].dtype) for k in top_ckpts[0][1]}
    cur_m = model.module if hasattr(model, 'module') else model
    cur_m.load_state_dict(avg_state); model.eval()

    val_logits, test_logits = [], []
    with torch.no_grad():
        for batch in val_loader:
            inp = {k: v.to(DEVICE) for k, v in batch.items()}; inp.pop('labels', None)
            with autocast(enabled=USE_AMP): out = model(**inp)
            val_logits.extend(out.logits.float().cpu().numpy())
        for batch in test_loader:
            inp = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast(enabled=USE_AMP): out = model(**inp)
            test_logits.extend(out.logits.float().cpu().numpy())

    del model, optimizer, scheduler, scaler, top_ckpts, avg_state
    gc.collect(); torch.cuda.empty_cache()
    return np.array(val_logits), np.array(test_logits), best_f1


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_logits      = np.zeros((len(y_all), NUM_LABELS))
test_logits_sum = np.zeros((len(texts_test), NUM_LABELS))
fold_scores     = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
    vl, tl, bf = train_fold(fold_idx, train_idx, val_idx)
    oof_logits[val_idx] = vl
    test_logits_sum    += tl
    fold_scores.append(bf)
    print(f'  Fold {fold_idx+1} best: {bf:.4f} | Elapsed: {(time.time()-t_start)/60:.1f}min')

test_logits_fold = test_logits_sum / N_FOLDS
print(f'\n{N_FOLDS}-FOLD: {[f"{s:.4f}" for s in fold_scores]}')
print(f'Mean: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')


In [ ]:
# ============================================================
# FULL-DATA MODEL  (blended with fold avg for test)
# ============================================================
def train_full_data():
    gc.collect(); torch.cuda.empty_cache()
    print(f'\n{"="*50}\nFULL-DATA MODEL | {len(y_all)} samples | {FULL_EPOCHS} epochs\n{"="*50}')

    all_idx  = np.arange(len(y_all))
    tr_load  = DataLoader(TextDataset(train_enc, y_all, all_idx),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=True, drop_last=True)
    te_load  = DataLoader(TextDataset(test_enc),
        batch_size=BATCH_SIZE*2, num_workers=NUM_WORKERS, pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
    base = model.model if hasattr(model, 'model') else model
    base.gradient_checkpointing_enable()
    if N_GPUS > 1: model = nn.DataParallel(model)
    model.to(DEVICE)

    fold_counts = np.bincount(y_all, minlength=NUM_LABELS).astype(float)
    criterion   = LDAMLoss(fold_counts, drw_epoch=1)
    criterion.to(DEVICE)

    actual_model = model.module if hasattr(model, 'module') else model
    optimizer = get_optimizer(actual_model, lr=LR * 0.8, llrd=0.9)
    total_steps = len(tr_load) * FULL_EPOCHS
    scheduler = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    scaler = GradScaler(enabled=USE_AMP)
    swa_states = []

    for epoch in range(FULL_EPOCHS):
        t0 = time.time(); criterion.set_epoch(epoch)
        model.train(); total_loss, valid_steps = 0.0, 0
        for batch in tqdm(tr_load, desc=f'Full E{epoch+1}', leave=False):
            optimizer.zero_grad()
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')
            with autocast(enabled=USE_AMP):
                out  = model(**inputs)
                loss = criterion(out.logits, labels)
            if not (torch.isnan(loss) or torch.isinf(loss)):
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                total_loss += loss.item(); valid_steps += 1
            scheduler.step()
        if epoch >= FULL_EPOCHS - 3:
            cur_m = model.module if hasattr(model, 'module') else model
            swa_states.append({k: v.cpu().clone() for k, v in cur_m.state_dict().items()})
        print(f'  Full E{epoch+1}/{FULL_EPOCHS} | loss={total_loss/max(valid_steps,1):.4f} | {time.time()-t0:.0f}s')

    avg_state = {k: (sum(s[k].float() for s in swa_states)/len(swa_states)
                     ).to(swa_states[0][k].dtype) for k in swa_states[0]}
    actual_model = model.module if hasattr(model, 'module') else model
    actual_model.load_state_dict(avg_state); model.eval()

    test_logits = []
    with torch.no_grad():
        for batch in te_load:
            inp = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast(enabled=USE_AMP): out = model(**inp)
            test_logits.extend(out.logits.float().cpu().numpy())

    del model, optimizer, scheduler, scaler, swa_states, avg_state
    gc.collect(); torch.cuda.empty_cache()
    return np.array(test_logits)

full_test_logits = train_full_data()
print(f'Full-data done | Total: {(time.time()-t_start)/60:.1f}min')


In [ ]:
# ============================================================
# TEMPERATURE CALIBRATION — conservative range [0.5, 2.0]
# OLD: [0.3, 3.0] → found T=0.30 (boundary = OOF overfit)
# NEW: [0.5, 2.0] → avoids hitting the boundary, more robust
# ============================================================
logits_t = torch.tensor(oof_logits, dtype=torch.float32)
labels_t = torch.tensor(y_all,     dtype=torch.long)

best_temp, best_nll = 1.0, float('inf')
for temp in np.arange(0.5, 2.0, 0.05):    # conservative [0.5, 2.0]
    nll = F.cross_entropy(logits_t / temp, labels_t).item()
    if nll < best_nll:
        best_nll = nll; best_temp = temp

print(f'Optimal temperature: {best_temp:.2f}  (searched [0.5, 2.0])')

# Blend fold avg (60%) + full-data (40%) for test
oof_probs       = torch.softmax(logits_t / best_temp, dim=-1).numpy()
fold_test_probs = torch.softmax(torch.tensor(test_logits_fold,   dtype=torch.float32) / best_temp, dim=-1).numpy()
full_test_probs = torch.softmax(torch.tensor(full_test_logits,   dtype=torch.float32) / best_temp, dim=-1).numpy()
test_probs = 0.6 * fold_test_probs + 0.4 * full_test_probs

oof_f1 = f1_score(y_all, oof_probs.argmax(1), average='macro')
print(f'OOF F1 (ModernBERT): {oof_f1:.4f}')
print(classification_report(y_all, oof_probs.argmax(1),
      target_names=[LABEL_MAP[i] for i in range(NUM_LABELS)]))


In [ ]:
# ============================================================
# SVC BASELINE
# ============================================================
from sklearn.model_selection import StratifiedKFold as SKF2
char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                              max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True)
word_tfidf = TfidfVectorizer(analyzer='word',    ngram_range=(1, 2),
                              max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True)
char_train = char_tfidf.fit_transform(texts_train)
char_test  = char_tfidf.transform(texts_test)
word_train = word_tfidf.fit_transform(texts_train)
word_test  = word_tfidf.transform(texts_test)
X_sparse_train = sparse_hstack([char_train, word_train])
X_sparse_test  = sparse_hstack([char_test,  word_test])

svc_oof   = np.zeros((len(y_all), NUM_LABELS))
svc_test  = np.zeros((len(texts_test), NUM_LABELS))
for fi, (tr, va) in enumerate(SKF2(n_splits=5, shuffle=True, random_state=SEED)
                               .split(np.zeros(len(y_all)), y_all)):
    clf = CalibratedClassifierCV(
        LinearSVC(C=5.0, class_weight='balanced', max_iter=20000), cv=3, method='sigmoid')
    clf.fit(X_sparse_train[tr], y_all[tr])
    svc_oof[va] = clf.predict_proba(X_sparse_train[va])
    svc_test   += clf.predict_proba(X_sparse_test) / 5
    print(f'  SVC fold {fi+1}: F1={f1_score(y_all[va], clf.predict(X_sparse_train[va]), average="macro"):.4f}')

svc_f1 = f1_score(y_all, svc_oof.argmax(1), average='macro')
print(f'SVC OOF F1: {svc_f1:.4f}')


In [ ]:
# ============================================================
# GLOBAL ENSEMBLE  — no per-class Nelder-Mead (avoids extra overfit)
# Simple grid search over alpha in [0, 1]
# ============================================================
best_alpha, best_blend_f1 = 0.85, 0.0
for alpha in np.arange(0.50, 1.01, 0.01):
    blend = alpha * oof_probs + (1 - alpha) * svc_oof
    f1    = f1_score(y_all, blend.argmax(1), average='macro')
    if f1 > best_blend_f1:
        best_blend_f1 = f1; best_alpha = alpha

oof_blend  = best_alpha * oof_probs  + (1 - best_alpha) * svc_oof
test_blend = best_alpha * test_probs + (1 - best_alpha) * svc_test

print(f'Optimal alpha: MB={best_alpha:.2f}, SVC={1-best_alpha:.2f}')
print(f'ModernBERT OOF F1: {oof_f1:.4f}')
print(f'SVC OOF F1:        {svc_f1:.4f}')
print(f'Blended OOF F1:    {best_blend_f1:.4f}')


In [ ]:
# ============================================================
# CONSERVATIVE THRESHOLD SWEEP  [0.85, 1.20]  passes=5
# Original winning range — avoids DeepSeek=1.485 overfit
# ============================================================
def threshold_sweep(probs, labels, lo=0.85, hi=1.20, steps=40, passes=5):
    best_mult = np.ones(NUM_LABELS)
    best_f1   = f1_score(labels, probs.argmax(1), average='macro')
    for _ in range(passes):
        improved = False
        for cls in range(NUM_LABELS):
            for m in np.linspace(lo, hi, steps):
                trial      = best_mult.copy(); trial[cls] = m
                f1 = f1_score(labels, (probs * trial).argmax(1), average='macro')
                if f1 > best_f1 + 1e-6:
                    best_f1 = f1; best_mult = trial.copy(); improved = True
        if not improved: break
    return best_mult, best_f1

thresholds, thresh_f1 = threshold_sweep(oof_blend, y_all)
print(f'OOF F1 after threshold: {thresh_f1:.4f}  (was {best_blend_f1:.4f})')
print('Multipliers:', {LABEL_MAP[i]: round(thresholds[i], 3) for i in range(NUM_LABELS)})

final_preds = (test_blend * thresholds).argmax(1)


In [ ]:
# ============================================================
# SUBMISSION
# ============================================================
id_col = None
for c in test_df.columns:
    if c.lower() in ('id', 'unnamed: 0', 'index'): id_col = c; break

pred_labels = [LABEL_MAP[p] for p in final_preds]
submission  = pd.DataFrame({'id': test_df[id_col], 'LABEL': pred_labels}) if id_col \
              else pd.DataFrame({'LABEL': pred_labels})
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
for i, nm in LABEL_MAP.items():
    cnt = (final_preds == i).sum()
    print(f'  {nm:10s}: {cnt} ({cnt/len(final_preds)*100:.1f}%)')

print(f'\n{"="*50}')
print(f'FINAL SUMMARY (v7 — conservative)')
print(f'  ModernBERT OOF F1: {oof_f1:.4f}')
print(f'  SVC OOF F1:        {svc_f1:.4f}')
print(f'  Blended OOF F1:    {best_blend_f1:.4f}  (MB={best_alpha:.2f})')
print(f'  + threshold nudge: {thresh_f1:.4f}  (range [0.85, 1.20])')
print(f'  Temperature:       {best_temp:.2f}  (range [0.5, 2.0])')
print(f'  DRW cap: {DRW_CAP}x  |  Folds: {N_FOLDS}  |  SWA top-{TOP_K_CKPTS}')
print(f'  Total time:        {(time.time()-t_start)/60:.1f} min')
print(f'{"="*50}')
print('Submission saved!')
